## Exploring predictions

In [1]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import sys
import gc
import random
import importlib as imp
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import silence_tensorflow
import seaborn as sns

import experiment_settings
import build_model
import plots
import build_data
import read_landsat
import methods
import methods
import predictions
import read_landsat

from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
# import rasterio
# tif_filename = "data/hii_coastal_buffer_mask.tif"
# tiffile = rasterio.open(tif_filename)
# tiff_data = tiffile.read()
# meta_data = tiffile.meta.copy()
# # print(meta_data)
# print(tiffile.block_shapes)
# tiff_data = tiffile.read()

# tif_filename = "data/landsat_export_1x1/landsat_40lat_70lon_2021.tif"
# tiffile = rasterio.open(tif_filename)
# tiff_data = tiffile.read()

# meta_data = tiffile.meta.copy()
# print(meta_data)
# print(tiffile.block_shapes)

# meta_data = tiffile.meta.copy()
# meta_data.update({"tiled": False,
#                   "compress": "lzw",
#                   "BIGTIFF": "yes",
#                 })
# # meta_data.update({"tiled": True,
# #                   "blockxsize": 512,
# #                   "blockysize": 512,
# #                   "compress": "lzw",
# #                   "BIGTIFF": "yes",
# #                 })
# with rasterio.open("data/test.tif", "w", **meta_data) as dst:
#     dst.write(tiff_data)

# new_tiffile = rasterio.open("data/test.tif")
# print(new_tiffile.meta)
# print(new_tiffile.block_shapes)

In [ ]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
print(f"tensorflow version = {tf.__version__}")

# tf.config.set_visible_devices([], "GPU")  # turn-off tensorflow-metal if it is on
print(tf.config.list_physical_devices("GPU"))

python version = 3.10.10 | packaged by conda-forge | (main, Mar 24 2023, 20:12:31) [Clang 14.0.6 ]
numpy version = 1.23.2
tensorflow version = 2.10.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# GET SETTINGS
EXP_NAME = "exp7"
REWRITE = False

settings = experiment_settings.get_settings(EXP_NAME)
settings["batch_size"] = 256
settings["mode"] = "inference"

directory_paths = methods.get_directories()
SAVE_MODEL_DIRECTORY = directory_paths["save_model_dir"]
DATA_DIRECTORY = directory_paths["data_dir"]
FIGURE_DIRECTORY = directory_paths["figures_dir"]
PREDICTIONS_DIRECTORY = directory_paths["predictions_dir"]
LANDSAT_DIRECTORY = directory_paths["landsat_dir"]
MOSAICS_DIRECTORY = directory_paths["mosaics_dir"]

In [ ]:
# GET THE DATA
imp.reload(build_data)
imp.reload(read_landsat)
imp.reload(methods)
imp.reload(predictions)

# for debugging
# settings["inference_region"] = (-7, -6, 106, 107)
settings["inference_region"] = (31, 32, 71, 72)

# set and get important settings
(lat_s_bound, lat_n_bound, lon_w_bound, lon_e_bound) = read_landsat.get_landsat_bounds(
    settings, region=settings["inference_region"]
)

# load the model
model = build_model.get_model(settings)

for year in np.arange(2021, 2023):  # 2000, 2005, 2007, 2010, 2021, 2022):
    print(" --- " + str(year) + "---")
    settings["inference_years"] = (year,)
    filenames_list = []

    for latfile in np.arange(
        lat_s_bound + settings["tile_len_deg"],
        lat_n_bound + settings["tile_len_deg"],
        settings["tile_len_deg"],
    ):
        for lonfile in np.arange(lon_w_bound, lon_e_bound, settings["tile_len_deg"]):
            # CHECK IF LANDSAT FILE EXISTS
            settings["tile"] = (
                latfile - settings["tile_len_deg"],
                latfile,
                lonfile,
                lonfile + settings["tile_len_deg"],
            )
            landsat_file = read_landsat.get_input_filename(
                settings["inference_years"], (latfile,), (lonfile,), settings
            )

            # CHECK IF LANDSAT FILE EXISTS
            if os.path.isfile(LANDSAT_DIRECTORY + landsat_file[0] + ".tif") is False:
                continue

            # TODO: check if landsat tile is all water, if so, create prediction file of all NODATA

            # CHECK IF PREDICTION FILE ALREADY EXISTS
            predictions_filename = (
                settings["exp_name"] + "_predictions_" + landsat_file[0]
            )
            filenames_list.append(PREDICTIONS_DIRECTORY + predictions_filename + ".tif")
            if (
                os.path.isfile(PREDICTIONS_DIRECTORY + predictions_filename + ".tif")
                and REWRITE is False
            ):
                continue
            print(landsat_file[0])

            # GET THE SAMPLE TAGS
            tags_inf, __ = build_data.get_tags(settings)
            if len(tags_inf[0]) == 0:
                continue

            # raise ValueError("STOP HERE")
            # BUILD THE DATA AND MAKE THE PREDICTIONS
            # tfds_inf = build_data.build_tf_dataset(
            #     settings, tags_inf, settings["batch_size"]
            # )
            # tfds_inf = tfds_inf.prefetch(tf.data.AUTOTUNE)

            # MAKE PREDICTIONS and SAVE AS TIF
            hfi_predict, hfi_labels, latlon_bounds = predictions.make_predictions(
                settings, model, tags_inf
            )

            # raise ValueError("STOP HERE")

            meta_data = predictions.save_predictions_tif(
                hfi_predict,
                PREDICTIONS_DIRECTORY + predictions_filename + ".tif",
                latlon_bounds=latlon_bounds,
            )

            filenames_list.append(PREDICTIONS_DIRECTORY + predictions_filename + ".tif")
            print("\n")

    # TILE THE PREDICTIONS TOGETHER
    mosaic_filename = (
        MOSAICS_DIRECTORY
        + settings["exp_name"]
        + "_"
        + str(settings["inference_years"][0])
        + "_mlhfi_mosaic.tif"
    )
    mosaic, mosaic_trans = predictions.create_mosaic(filenames_list)
    meta_data = predictions.save_predictions_tif(
        mosaic, mosaic_filename, trans=mosaic_trans
    )
    print("mosaic saved.")

Metal device set to: Apple M1 Max

systemMemory: 64.00 GB
maxCacheSize: 24.00 GB

 --- 2021---
landsat_40lat_70lon_2021
output region shape = (742, 742)
n_inference = (550564,)


ValueError: STOP HERE

In [ ]:
raise ValueError("STOP HERE")

In [ ]:
inc = 1
tags_inf_part = (
    tags_inf[0][::inc],
    tags_inf[1][::inc],
    tags_inf[2][::inc],
    tags_inf[3][::inc],
)
print(tags_inf_part[0].shape)

(3441025,)


In [ ]:
# BUILD THE DATA AND MAKE THE PREDICTIONS
tfds_inf = build_data.build_tf_dataset(settings, tags_inf_part, settings["batch_size"])

# PREDICT
hfi_predict = model.predict(tfds_inf, verbose=0)
gc.collect()

1321

In [ ]:
imp.reload(build_data)
tags = tags_inf_part
tags_dict = {}
for filename in np.unique(tags[-1]):
    isample = [index for (index, item) in enumerate(tags[-1]) if item == filename]
    tags_dict[filename] = np.asarray(isample)

data_gen = build_data.data_generator(settings, tags, tags_dict)

In [ ]:
imp.reload(build_data)

hfi_predict = np.zeros((tags[0].shape[0],1))

inc = 5_000
for i in np.arange(0, tags[0].shape[0], inc):
    index_end = np.min([i + inc, tags[0].shape[0]])
    x_input, y_output = data_gen.get_data(np.arange(i, index_end))
    hfi_predict[i:index_end] = model.predict(x_input, batch_size=settings["batch_size"], verbose=0)
    gc.collect()


KeyboardInterrupt: 

In [ ]:
hfi_predict.shape